# 03 — Evaluate on MIDI-Instruct (Colab T4)

Run **infer → decode → score** on the test split. Compare LoRA adapter vs zero-shot base model vs copy-source baseline.

In [ ]:
# @title Configuration
WORK = "/content"
MIDI_LLM = f"{WORK}/midi-llm"
MUSICINSTRUCT = f"{WORK}/musicinstruct"
MANIFEST = f"{MUSICINSTRUCT}/data/pilot/pilot.jsonl"
RUN_ID = "pilot_v1"
RUNS_ROOT = f"{WORK}/runs/{RUN_ID}"

REPR = "remi"  # @param ["remi", "octuple", "amt"]
SPLIT = "test"  # @param ["test", "validation"]
MAX_ITEMS = None  # @param {type:"raw"}  # e.g. 5 for smoke test
MAX_NEW_TOKENS = 512  # @param {type:"integer"}
SEED = 42  # @param {type:"integer"}
SCORE_TIMEOUT = 600  # @param {type:"integer"}

MODE = "lora"  # @param ["lora", "zero_shot", "copy_source"]
ADAPTER_DIR = None  # @param {type:"raw"}  # default: RUNS_ROOT/REPR/lora_adapter

USE_DRIVE = False
DRIVE_RUNS = "/content/drive/MyDrive/midi-llm/runs"

In [ ]:
# @title Setup
import os
from pathlib import Path

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUNS_ROOT = f"{DRIVE_RUNS}/{RUN_ID}"

%cd {MIDI_LLM}
!pip install -q -e ".[repr,train]"
!pip install -q -e "{MUSICINSTRUCT}"
!pip install -q "git+https://github.com/jthickstun/anticipation.git" mido

if not os.environ.get("HF_TOKEN") and MODE != "copy_source":
    from getpass import getpass
    from huggingface_hub import login
    os.environ["HF_TOKEN"] = getpass("HF token: ")
    login(token=os.environ["HF_TOKEN"])

adapter = ADAPTER_DIR or f"{RUNS_ROOT}/{REPR}/lora_adapter"
eval_dir = f"{RUNS_ROOT}/eval_{REPR}_{MODE}_{SPLIT}"
print("Eval output:", eval_dir)

In [ ]:
# @title Run evaluation
import json
from pathlib import Path
from midi_llm.eval.edit_pipeline import copy_source_as_baseline, run_edit_eval

Path(eval_dir).mkdir(parents=True, exist_ok=True)

if MODE == "copy_source":
    preds = f"{eval_dir}/predictions.jsonl"
    copy_result = copy_source_as_baseline(MANIFEST, preds, split=SPLIT)
    from midi_llm.eval.musicinstruct_runner import run_musicinstruct_eval
    results_path = f"{eval_dir}/results.json"
    results = run_musicinstruct_eval(
        MANIFEST,
        preds,
        output_results=results_path,
        split=SPLIT,
        timeout_sec=SCORE_TIMEOUT,
    )
    summary = {
        "mode": MODE,
        "n_predictions": copy_result.n_written,
        "n_copy_failed": copy_result.n_failed,
        "overall": results.get("overall"),
    }
elif MODE == "zero_shot":
    summary = run_edit_eval(
        MANIFEST,
        eval_dir,
        repr_name=REPR,
        split=SPLIT,
        adapter_path=None,
        max_items=MAX_ITEMS,
        max_new_tokens=MAX_NEW_TOKENS,
        seed=SEED,
        score_timeout_sec=SCORE_TIMEOUT,
    )
else:
    if not Path(adapter).is_dir():
        raise FileNotFoundError(f"Adapter not found: {adapter}")
    summary = run_edit_eval(
        MANIFEST,
        eval_dir,
        repr_name=REPR,
        split=SPLIT,
        adapter_path=adapter,
        max_items=MAX_ITEMS,
        max_new_tokens=MAX_NEW_TOKENS,
        seed=SEED,
        score_timeout_sec=SCORE_TIMEOUT,
    )

print(json.dumps(summary, indent=2))

In [ ]:
# @title Results table
import json
from pathlib import Path

results_path = Path(eval_dir) / "results.json"
if results_path.is_file():
    data = json.loads(results_path.read_text())
    overall = data.get("overall", {})
    print("Overall metrics:")
    for k, v in overall.items():
        print(f"  {k}: {v}")
    print("\nBy op_family:")
    for op, metrics in data.get("by_op_family", {}).items():
        print(f"  {op}: joint={metrics.get('joint')} edit={metrics.get('edit_success')} preserve={metrics.get('preserve')}")
else:
    print("No results.json — check eval_summary.json")
    print((Path(eval_dir) / "eval_summary.json").read_text())

In [ ]:
# @title Compare all active representations (LoRA adapters must exist)
RUN_ALL_REPRS = False  # @param {type:"boolean"}

if RUN_ALL_REPRS:
    import json
    rows = []
    for repr_name in ["remi", "octuple", "amt"]:
        ad = f"{RUNS_ROOT}/{repr_name}/lora_adapter"
        out = f"{RUNS_ROOT}/eval_{repr_name}_lora_{SPLIT}"
        if not Path(ad).is_dir():
            print("skip", repr_name, "(no adapter)")
            continue
        s = run_edit_eval(
            MANIFEST,
            out,
            repr_name=repr_name,
            split=SPLIT,
            adapter_path=ad,
            max_items=MAX_ITEMS,
            seed=SEED,
            score_timeout_sec=SCORE_TIMEOUT,
        )
        o = s.get("overall") or {}
        rows.append({"repr": repr_name, **{k: o.get(k) for k in ("joint", "edit_success", "preserve", "validity")}})
    print(json.dumps(rows, indent=2))

In [ ]:
# @title Download eval artifacts
import shutil
from pathlib import Path

zip_base = f"{WORK}/eval_{RUN_ID}_{REPR}_{MODE}"
shutil.make_archive(zip_base, "zip", eval_dir)
try:
    from google.colab import files
    files.download(zip_base + ".zip")
except ImportError:
    print("Zip at:", zip_base + ".zip")